# Train one YOLO model on Colab

Trains a single model variant end-to-end and persists the run folder to Google Drive.
Run one model per Colab session (free tier ~12h is not enough for all variants at once).

**Before running:** push this repo to GitHub and set `REPO_URL` below.
Also upload `dataset.zip` + `dataset.meta.json` to `MyDrive/yolo-pipeline/datasets/v1/`.

In [ ]:
# === EDIT THESE PER SESSION ===
REPO_URL    = "https://github.com/<your-user>/<your-repo>.git"
REPO_BRANCH = "main"
DATASET_VERSION = "v1"
MODEL       = "yolo11s"                  # one of: yolo11n yolo11s yolo11m yolo11l yolo12s
CONFIG      = "configs/yolo11s.yaml"     # matching variant config
RESUME      = False                      # set True to continue the latest Drive run for MODEL
# ==============================

In [ ]:
# 1. Mount Drive + GPU check
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi

In [ ]:
# 2. Clone repo (fresh each session so code is exactly what's on GitHub)
import shutil, os
if os.path.isdir('/content/code'):
    shutil.rmtree('/content/code')
!git clone --branch {REPO_BRANCH} {REPO_URL} /content/code
%cd /content/code
!git rev-parse HEAD

In [ ]:
# 3. Install pinned deps
!pip install -q -r requirements.txt
import ultralytics, torch
print('ultralytics', ultralytics.__version__, '| torch', torch.__version__, '| cuda', torch.version.cuda)

In [ ]:
# 4. Materialize dataset (unzip + sha256 verify; idempotent)
from pipeline.dataset import ensure_dataset
from pipeline import paths

data_yaml = ensure_dataset(
    zip_path=paths.dataset_zip(DATASET_VERSION),
    meta_path=paths.dataset_meta(DATASET_VERSION),
    target=paths.LOCAL_DATASET,
)
print('data.yaml:', data_yaml)

In [ ]:
# 5. Clean up any leftover .tmp/.stale on Drive from a previous failed copy
from pipeline.persist import cleanup_tmp
cleanup_tmp(paths.RUNS_DIR)

In [ ]:
# 6. Train
from pipeline.train import run as train_run
run_dir = train_run(
    model=MODEL,
    config=CONFIG,
    drive_runs_dir=paths.RUNS_DIR,
    local_runs_dir=paths.LOCAL_RUNS,
    data_yaml=data_yaml,
    dataset_meta_path=paths.dataset_meta(DATASET_VERSION),
    base_config='configs/base.yaml',
    resume=RESUME,
)
print('Drive run dir:', run_dir)

In [ ]:
# 7. Held-out test-set evaluation (always split='test', not the default 'val')
from pipeline.evaluate import run as eval_run
eval_payload = eval_run(
    run_dir=run_dir,
    data_yaml=data_yaml,
    drive_runs_dir=paths.RUNS_DIR,
)
print('test mAP50:', eval_payload['overall']['mAP50'])
print('test mAP50-95:', eval_payload['overall']['mAP50_95'])